# AMATH141 LAB5 — Monomial form foundations

Build the core tools for interpolation in the monomial basis: the Vandermonde matrix, the polynomial coefficients, and Horner evaluation.

Coefficient order is `[a0, a1, ..., an]` so that
$$P_n(x) = a_0 + a_1 x + a_2 x^2 + \cdots + a_n x^n.$$

## Question 1 — Build the Vandermonde matrix

**Question.** Given a list of distinct nodes `x = [x0, x1, ..., xn]`, return the matrix

$$V_{i,j} = x_i^{\,j}$$

as a list of rows. For example, if `x = [0, 1, 2]`, the result should be

```
[
    [1, 0, 0],
    [1, 1, 1],
    [1, 2, 4],
]
```

In [ ]:
def vandermonde_matrix(x):
    # return the Vandermonde matrix as a list of lists
    n = len(x)
    matrix = []
    for xi in x:
        row = []
        for j in range(n):
            row.append(xi**j)
        matrix.append(row)
    return matrix

# Sanity check
print(vandermonde_matrix([0, 1, 2]))
# Expected: [[1, 0, 0], [1, 1, 1], [1, 2, 4]]

## Question 2 — Recover monomial coefficients

**Question.** Given interpolation nodes `x` and values `y`, solve the Vandermonde system to recover the coefficients `[a0, ..., an]` of the interpolating polynomial

$$P_n(x) = a_0 + a_1 x + \cdots + a_n x^n.$$

Return the coefficients in increasing-power order as a Python list.

In [ ]:
import numpy as np

def monomial_coefficients(x, y):
    n = len(x)
    V = [[xi**j for j in range(n)] for xi in x]
    a = np.linalg.solve(np.array(V, dtype=float), np.array(y, dtype=float))
    return a.tolist()

# Sanity checks
print(monomial_coefficients([0, 1, 2], [1, 3, 7]))   # quadratic x^2 + x + 1 -> [1, 1, 1]
print(monomial_coefficients([-1, 0, 1], [1, 0, 1]))  # pure square x^2     -> [0, 0, 1]

## Question 3 — Evaluate with Horner's method

**Question.** Given coefficients `[a0, a1, ..., an]` in increasing-power order, evaluate

$$P_n(t) = a_0 + a_1 t + \cdots + a_n t^n$$

using Horner's algorithm.

Horner's method rewrites the polynomial in nested form

$$P_n(t) = a_0 + t\bigl(a_1 + t\bigl(a_2 + \cdots + t\,a_n\bigr)\bigr),$$

which evaluates $P_n(t)$ in only $n$ multiplications and $n$ additions.

In [ ]:
def horner_eval(coeffs, t):
    # evaluate the polynomial at t using Horner's method
    result = 0.0
    for a in reversed(coeffs):
        result = result * t + a
    return result

# Sanity checks
print(horner_eval([1, 1, 1], 2))   # 1 + 1*2 + 1*4 = 7
print(horner_eval([0, 0, 1], 3))   # 0 + 0 + 9    = 9
print(horner_eval([1, -2, 3], 0))  # 1            = 1

## End-to-end check

Combine all three functions to interpolate a known polynomial and verify exact recovery.

In [ ]:
# Take the polynomial Q(x) = 2 - x + 0.5 x^2 sampled at three nodes
x_nodes = [0, 1, 2]
y_vals  = [2 - xi + 0.5*xi**2 for xi in x_nodes]

coeffs = monomial_coefficients(x_nodes, y_vals)
print("Recovered coefficients:", coeffs)  # should be approximately [2.0, -1.0, 0.5]

# Evaluate at a few new points and compare
for t in [0.5, 1.5, 3.0]:
    p_horner = horner_eval(coeffs, t)
    p_true   = 2 - t + 0.5*t**2
    print(f"t = {t}: Horner = {p_horner:.6f}, true = {p_true:.6f}")